# OpenAI Embeddings와 캐싱 최적화

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- 임베딩(Embedding)이 무엇인지, 왜 RAG에서 핵심인지 설명할 수 있어요
- `OpenAIEmbeddings`로 텍스트를 벡터로 변환하고 유사도를 계산할 수 있어요
- `CacheBackedEmbeddings`로 API 비용을 절감할 수 있어요

## 사전 지식

- 6-1 DocumentLoaders, 6-2 Chunking 학습 완료
- OpenAI API 키 설정 (`.env` 파일)

---

## 임베딩(Embedding)이란?

임베딩은 텍스트를 숫자 배열(벡터)로 변환하는 과정이에요. 비슷한 의미의 문장은 벡터 공간에서도 가까이 위치하기 때문에, 검색 시 의미 기반 매칭이 가능해요.

```mermaid
flowchart LR
    T["텍스트<br/>&#34;안녕하세요&#34;"]:::input
    M["임베딩 모델<br/>text-embedding-3-small"]:::process
    V["벡터<br/>0.1, 0.5, -0.3, ...<br/>1536차원"]:::output

    T --> M --> V

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## OpenAI Embeddings 모델 비교

| 모델 | 차원 | 가격 (백만 토큰당) | MTEB 성능 | 용도 |
|------|------|-------------------|-----------|------|
| `text-embedding-3-small` | 1536 | $0.02 | 62.3% | 일반 용도, 비용 효율적 |
| `text-embedding-3-large` | 3072 | $0.13 | 64.6% | 고품질 필요 시 |
| `text-embedding-ada-002` | 1536 | $0.10 | 61.0% | 레거시 모델 |

> **실무 팁**: 대부분의 RAG 프로젝트에서는 `text-embedding-3-small`로 충분해요. 비용이 `ada-002`의 1/5이면서 성능은 오히려 더 좋아요.

## 환경 설정

`.env` 파일에 `OPENAI_API_KEY`가 설정되어 있어야 해요.

In [1]:
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

---

## 1. OpenAI Embeddings 기본 사용법

`OpenAIEmbeddings`를 초기화해볼게요. 사용할 모델만 지정하면 바로 쓸 수 있어요.

In [2]:

# ---------------------------------------------------
# OpenAIEmbeddings 모델 초기화
# ---------------------------------------------------

from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings: OpenAI API를 통해 텍스트를 벡터로 변환하는 클래스
# model: 사용할 임베딩 모델 지정 (text-embedding-3-small 권장)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


### 1.1 단일 쿼리 임베딩

`embed_query()` 메서드는 단일 텍스트를 임베딩 벡터로 변환해요.

주로 사용자의 **검색 질문(Query)**을 벡터로 변환할 때 사용해요.

> 🎯 **강의 포인트**: `embed_query()`는 **검색 질문용**, `embed_documents()`는 **문서 저장용**이에요. 이 둘을 바꿔 쓰면 검색 정확도가 떨어질 수 있어요 — 일부 모델은 쿼리와 문서에 서로 다른 임베딩 방식을 적용하거든요.

In [3]:

# ---------------------------------------------------
# 단일 쿼리 텍스트를 임베딩 벡터로 변환
# ---------------------------------------------------

# ============================================================
# TODO: 아래 쿼리 텍스트를 벡터로 변환해보세요
# 힌트: embed_query() 메서드는 단일 텍스트를 입력받아 벡터 리스트를 반환해요
# 예상 결과: 1536차원의 float 리스트 출력
# ============================================================

# 검색할 쿼리 텍스트
query_text = "우주 탐사의 미래는 어떻게 될까요?"

# 1단계: 쿼리를 벡터로 변환
# embed_query(): 검색 질문(Query) 전용 임베딩 메서드
query_vector = embeddings.embed_query(query_text)

print(f"원본 텍스트: {query_text}")
print(f"벡터 차원: {len(query_vector)}")
print(f"벡터 일부: {query_vector[:5]}")


원본 텍스트: 우주 탐사의 미래는 어떻게 될까요?
벡터 차원: 1536
벡터 일부: [0.023040771484375, -0.0179901123046875, 0.03350830078125, 0.0009665489196777344, -0.039764404296875]


### 1.2 여러 문서 임베딩

`embed_documents()` 메서드는 여러 텍스트를 한 번에 임베딩해요.

주로 **문서들을 벡터 DB에 저장**할 때 사용해요. 단건 호출보다 훨씬 효율적이에요.

> **주의**: `embed_query()`와 `embed_documents()`를 구분해서 써야 해요. 일부 모델은 쿼리와 문서에 다른 임베딩 방식을 적용하거든요.

In [4]:

# ---------------------------------------------------
# 여러 문서를 한 번에 임베딩 (배치 처리)
# ---------------------------------------------------

# ============================================================
# TODO: 아래 문서 리스트를 한 번에 벡터로 변환해보세요
# 힌트: embed_documents()는 리스트를 입력받아 벡터 리스트의 리스트를 반환해요
# 예상 결과: 4개 문서 × 1536차원 벡터 배열 출력
# ============================================================

# 벡터 DB에 저장할 문서들
documents = [
    "화성 탐사 로봇이 물의 흔적을 발견했습니다.",
    "SpaceX가 재사용 가능한 로켓 기술을 개발했습니다.",
    "제임스 웹 우주 망원경이 초기 우주의 모습을 촬영했습니다.",
    "국제 우주 정거장에서 새로운 실험이 진행 중입니다.",
]

# 1단계: 모든 문서를 한 번에 벡터로 변환 (배치 처리)
# embed_documents(): 문서 색인(저장) 전용 임베딩 메서드 — embed_query()와 구분해서 사용
doc_vectors = embeddings.embed_documents(documents)

print(f"변환된 문서 수: {len(doc_vectors)}")
print(f"각 벡터 차원: {len(doc_vectors[0])}")
print(f"\n첫 번째 문서: {documents[0]}")
print(f"벡터 일부: {doc_vectors[0][:5]}")


변환된 문서 수: 4
각 벡터 차원: 1536

첫 번째 문서: 화성 탐사 로봇이 물의 흔적을 발견했습니다.
벡터 일부: [-0.00827789306640625, 0.046478271484375, -0.03778076171875, 0.039459228515625, -0.001010894775390625]


### 1.3 차원 축소

`dimensions` 파라미터로 임베딩 벡터의 차원을 줄일 수 있어요.

벡터 DB 저장 공간을 절약하거나 검색 속도를 높이고 싶을 때 사용해요. 약간의 정확도 감소가 발생할 수 있어요.

> 💡 **실무 팁**: 수백만 개 이상의 대규모 문서를 처리할 때 차원 축소를 고려해보세요. 저장 비용과 검색 속도 모두 개선돼요.

> ⚠️ **자주 하는 실수**: 차원을 줄이면 정보 손실이 발생해요. 너무 낮은 차원(256 미만)으로 설정하면 검색 품질이 크게 떨어질 수 있으니 256~1024 범위에서 실험해보세요.

In [5]:

# ---------------------------------------------------
# dimensions 파라미터로 벡터 차원 축소
# ---------------------------------------------------

# ============================================================
# TODO: 1024차원으로 축소된 임베딩 모델을 만들고 비교해보세요
# 힌트: OpenAIEmbeddings의 dimensions 파라미터에 원하는 차원 수를 전달해요
# 예상 결과: 원본 1536차원 vs 축소 1024차원 비교 출력
# ============================================================

# 1단계: 차원 축소 모델 생성
# dimensions: 벡터 차원 수 지정 (기본 1536, 최소 256)
embeddings_small = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024  # 기본 1536에서 1024로 축소
)

# 2단계: 동일한 텍스트를 두 모델로 임베딩하여 차원 비교
sample_text = "인공지능이 의료 진단을 혁신하고 있습니다."

vector_original = embeddings.embed_query(sample_text)
vector_small = embeddings_small.embed_query(sample_text)

print(f"원본 모델 차원: {len(vector_original)}")
print(f"축소 모델 차원: {len(vector_small)}")
print(f"저장 공간 절약: {(1 - len(vector_small)/len(vector_original)) * 100:.1f}%")


원본 모델 차원: 1536
축소 모델 차원: 1024
저장 공간 절약: 33.3%


---

## 2. 유사도 계산과 검색

임베딩 벡터가 생성됐으면, 이제 쿼리와 문서 간의 **코사인 유사도(Cosine Similarity)**를 계산해서 가장 관련 있는 문서를 찾아볼게요.

코사인 유사도는 두 벡터가 이루는 각도를 기반으로 유사도를 계산해요. 값이 1에 가까울수록 유사한 문서예요.

> 🔑 **핵심 개념**: 코사인 유사도는 벡터의 **크기가 아닌 방향**을 비교해요. "나는 학생이다"와 "나는 대학생이다"는 길이가 달라도 의미가 비슷해서 높은 유사도를 가져요. RAG에서 검색 품질을 결정하는 가장 핵심적인 연산이에요.

In [6]:

# ---------------------------------------------------
# 코사인 유사도로 쿼리-문서 간 유사도 계산 및 검색
# ---------------------------------------------------

# ============================================================
# TODO: 쿼리와 문서 벡터 간 유사도를 계산하고 순위를 매겨보세요
# 힌트: cosine_similarity([query_vector], doc_vectors) → (1, n) 배열 반환
# 예상 결과: 유사도 순으로 정렬된 문서 목록 출력
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1단계: 검색 쿼리 및 문서 임베딩 준비
query = "화성 탐사에 대해 알려주세요"

query_vector = embeddings.embed_query(query)
doc_vectors = embeddings.embed_documents(documents)

# 2단계: 코사인 유사도 계산
# cosine_similarity: 두 벡터 사이의 각도 기반 유사도 (1에 가까울수록 유사)
similarities = cosine_similarity([query_vector], doc_vectors)[0]

# 3단계: 유사도 내림차순으로 정렬
ranked_indices = np.argsort(similarities)[::-1]

print(f"검색 쿼리: {query}")
print("=" * 60)
print("\n📊 검색 결과 (유사도 순):")
for i, idx in enumerate(ranked_indices, 1):
    print(f"\n[{i}위] 유사도: {similarities[idx]:.4f}")
    print(f"문서: {documents[idx]}")


검색 쿼리: 화성 탐사에 대해 알려주세요

📊 검색 결과 (유사도 순):

[1위] 유사도: 0.4771
문서: 화성 탐사 로봇이 물의 흔적을 발견했습니다.

[2위] 유사도: 0.1341
문서: 국제 우주 정거장에서 새로운 실험이 진행 중입니다.

[3위] 유사도: 0.1244
문서: 제임스 웹 우주 망원경이 초기 우주의 모습을 촬영했습니다.

[4위] 유사도: 0.1168
문서: SpaceX가 재사용 가능한 로켓 기술을 개발했습니다.


---

## 3. CacheBackedEmbeddings: 비용 절감의 핵심

**문제**: 동일한 텍스트를 반복해서 임베딩하면 API 비용이 계속 발생해요.

**해결책**: `CacheBackedEmbeddings`(캐시 기반 임베딩)를 사용하면 한 번 계산한 임베딩을 저장하고 재사용할 수 있어요.

```mermaid
flowchart TD
    Q["텍스트 입력"]:::input
    C{"캐시에<br/>존재?"}:::process
    Y["캐시에서 반환<br/>(API 호출 없음)"]:::output
    N["OpenAI API 호출<br/>→ 캐시 저장"]:::process
    R["벡터 반환"]:::output

    Q --> C
    C -->|"예"| Y --> R
    C -->|"아니오"| N --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

> 🎯 **강의 포인트**: 운영 환경에서 `CacheBackedEmbeddings` 없이 RAG를 배포하면 동일 문서를 반복 임베딩할 때마다 비용이 발생해요. 수천 개의 문서를 매일 재임베딩한다고 생각해보세요 — 캐싱은 선택이 아닌 **필수**예요.

### 3.1 LocalFileStore: 파일 시스템 캐싱

로컬 파일에 캐시를 저장하면 프로그램을 종료해도 캐시가 유지돼요.

In [7]:

# ---------------------------------------------------
# CacheBackedEmbeddings 생성 (LocalFileStore 사용)
# ---------------------------------------------------

# ============================================================
# TODO: 파일 캐시를 사용하는 임베딩 모델을 설정해보세요
# 힌트: CacheBackedEmbeddings.from_bytes_store(기반_모델, 저장소, namespace=...)
# 예상 결과: "캐시 기반 임베딩 모델 생성 완료" 출력
# ============================================================

from langchain.storage import LocalFileStore, InMemoryByteStore
from langchain.embeddings import CacheBackedEmbeddings

# 1단계: 기본 임베딩 모델 생성
base_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2단계: 로컬 파일 저장소 설정
# LocalFileStore: 임베딩 결과를 파일 시스템에 영구 저장
cache_store = LocalFileStore("./cache/embeddings/")

# 3단계: 캐시 기반 임베딩 생성
# CacheBackedEmbeddings: 동일 텍스트는 API 대신 캐시에서 반환
# namespace: 모델별로 캐시를 분리하는 접두어
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=base_embeddings,
    document_embedding_cache=cache_store,
    namespace="openai-text-embedding-3-small"  # 모델별로 구분
)

print("✅ 캐시 기반 임베딩 모델 생성 완료")
print(f"캐시 저장 위치: ./cache/embeddings/")

✅ 캐시 기반 임베딩 모델 생성 완료
캐시 저장 위치: ./cache/embeddings/


/Users/mhso/working/hankyung-llm-agent/02_langchain_basics/.venv/lib/python3.11/site-packages/langchain/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


### 3.2 캐싱 효과 확인

동일한 문서를 두 번 임베딩해서 속도 차이를 확인해볼게요.

In [8]:

# ---------------------------------------------------
# 첫 번째(API 호출)와 두 번째(캐시) 실행 속도 비교
# ---------------------------------------------------

import time

# 테스트용 문서 준비
test_documents = [
    "양자 컴퓨터가 암호화 기술에 미치는 영향을 연구하고 있습니다.",
    "신재생 에너지 발전소가 전 세계적으로 증가하고 있습니다.",
    "자율주행차 기술이 교통 시스템을 변화시키고 있습니다.",
    "유전자 편집 기술이 난치병 치료의 새로운 가능성을 열고 있습니다.",
    "블록체인 기술이 금융 산업의 투명성을 높이고 있습니다."
]

# 1단계: 첫 번째 실행 (API 호출 발생 — 캐시 없음)
print("🔄 첫 번째 실행: API 호출")
start_time = time.time()
vectors_1 = cached_embeddings.embed_documents(test_documents)
first_run_time = time.time() - start_time
print(f"⏱️  소요 시간: {first_run_time:.2f}초")

# 2단계: 두 번째 실행 (캐시에서 로드 — API 호출 없음)
print("\n⚡ 두 번째 실행: 캐시에서 로드")
start_time = time.time()
vectors_2 = cached_embeddings.embed_documents(test_documents)
second_run_time = time.time() - start_time
print(f"⏱️  소요 시간: {second_run_time:.2f}초")

# 3단계: 결과 비교
print("\n" + "=" * 60)
print("💡 캐싱 효과:")
print(f"  - 속도 향상: {first_run_time / second_run_time:.1f}배 빠름")
print(f"  - 비용 절감: 두 번째 실행은 무료 (API 호출 없음)")
print(f"  - 벡터 동일성: {np.allclose(vectors_1[0], vectors_2[0])}")


🔄 첫 번째 실행: API 호출
⏱️  소요 시간: 0.00초

⚡ 두 번째 실행: 캐시에서 로드
⏱️  소요 시간: 0.00초

💡 캐싱 효과:
  - 속도 향상: 3.6배 빠름
  - 비용 절감: 두 번째 실행은 무료 (API 호출 없음)
  - 벡터 동일성: True


### 3.3 실제 RAG 시나리오: 문서 로드 → 분할 → 벡터 DB 생성

실제 RAG 시스템에서 캐싱이 얼마나 유용한지 확인해볼게요.

In [9]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


# ---------------------------------------------------
# 실제 RAG 시나리오: 문서 로드 → 분할 → 벡터 DB 생성
# ---------------------------------------------------

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 1단계: 문서 로드
loader = TextLoader("./data/appendix-keywords.txt", encoding="utf-8")
raw_documents = loader.load()

# 2단계: 문서 분할
# chunk_size=500: 각 청크의 최대 글자 수
# chunk_overlap=50: 청크 간 겹치는 글자 수 (문맥 연속성 유지)
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
split_documents = text_splitter.split_documents(raw_documents)

print(f"📄 로드된 문서 수: {len(raw_documents)}")
print(f"✂️  분할된 청크 수: {len(split_documents)}")


📄 로드된 문서 수: 1
✂️  분할된 청크 수: 15


In [10]:
# 3단계: 캐싱 임베딩으로 벡터 DB 생성 (첫 실행)
print("\n🔄 벡터 DB 생성 (첫 실행 - API 호출)")
start_time = time.time()
vector_db = FAISS.from_documents(split_documents, cached_embeddings)
first_creation_time = time.time() - start_time
print(f"⏱️  소요 시간: {first_creation_time:.2f}초")



🔄 벡터 DB 생성 (첫 실행 - API 호출)
⏱️  소요 시간: 0.07초


In [11]:
# 4단계: 동일한 문서로 벡터 DB 재생성 (캐시 활용)
print("\n⚡ 벡터 DB 재생성 (캐시 활용 - API 호출 없음)")
start_time = time.time()
vector_db_2 = FAISS.from_documents(split_documents, cached_embeddings)
second_creation_time = time.time() - start_time
print(f"⏱️  소요 시간: {second_creation_time:.2f}초")

print("\n" + "=" * 60)
print("💰 비용 절감 효과:")
print(f"  - 속도 향상: {first_creation_time / second_creation_time:.1f}배")
print(f"  - 예상 비용 절감: 두 번째 실행은 $0 (API 호출 0회)")



⚡ 벡터 DB 재생성 (캐시 활용 - API 호출 없음)
⏱️  소요 시간: 0.00초

💰 비용 절감 효과:
  - 속도 향상: 14.4배
  - 예상 비용 절감: 두 번째 실행은 $0 (API 호출 0회)


### 3.4 InMemoryByteStore: 메모리 캐싱

`LocalFileStore`와 `InMemoryByteStore`(인메모리 저장소)의 차이를 살펴볼게요:

| 특성 | LocalFileStore | InMemoryByteStore |
|------|----------------|-------------------|
| 저장 위치 | 파일 시스템 | 메모리 |
| 영구성 | 영구 저장 | 프로그램 종료 시 삭제 |
| 속도 | 빠름 | 매우 빠름 |
| 용도 | 운영 환경 | 테스트·임시 작업 |

> ⚠️ **자주 하는 실수**: 테스트할 때 `InMemoryByteStore`를 사용했다가 운영 배포 때 그대로 쓰는 경우가 있어요. 운영 환경에서는 반드시 `LocalFileStore`(또는 Redis 등 외부 저장소)로 교체해야 서버 재시작 후에도 캐시가 유지돼요.

In [12]:

# ---------------------------------------------------
# InMemoryByteStore로 메모리 캐시 임베딩 생성
# ---------------------------------------------------

# 1단계: 메모리 기반 캐시 저장소 생성
# InMemoryByteStore: RAM에 저장, 프로그램 종료 시 삭제됨 (테스트·임시 작업용)
memory_store = InMemoryByteStore()

# 2단계: 메모리 캐싱 임베딩 생성
memory_cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=base_embeddings,
    document_embedding_cache=memory_store,
    namespace="openai-memory-cache"
)

# 3단계: 동작 확인
test_texts = [
    "메모리 캐싱 테스트 문장 1",
    "메모리 캐싱 테스트 문장 2"
]

# 첫 번째: API 호출
print("첫 번째 임베딩 (API 호출)")
vecs_1 = memory_cached_embeddings.embed_documents(test_texts)

# 두 번째: 메모리에서 로드
print("두 번째 임베딩 (메모리 캐시)")
vecs_2 = memory_cached_embeddings.embed_documents(test_texts)

print("\n⚠️  주의: 프로그램을 종료하면 메모리 캐시는 사라집니다!")


첫 번째 임베딩 (API 호출)
두 번째 임베딩 (메모리 캐시)

⚠️  주의: 프로그램을 종료하면 메모리 캐시는 사라집니다!


---

## 4. 캐시 저장소 관리

`yield_keys()`로 캐시에 저장된 항목들을 확인할 수 있어요.

In [13]:

# ---------------------------------------------------
# 캐시 저장소에 저장된 키 목록 확인
# ---------------------------------------------------

# yield_keys(): 저장소에 저장된 모든 캐시 키 반환
cached_keys = list(cache_store.yield_keys())

print(f"캐시된 항목 수: {len(cached_keys)}")
if cached_keys:
    print(f"\n첫 5개 캐시 키:")
    for key in cached_keys[:5]:
        print(f"  - {key}")


캐시된 항목 수: 20

첫 5개 캐시 키:
  - openai-text-embedding-3-small7e9823f0-0ca6-5136-af98-dc86886e4ea9
  - openai-text-embedding-3-small16a88813-7392-51e3-a520-a7882b25af2b
  - openai-text-embedding-3-small0673a452-0cf5-5a8f-b6a8-94239896ef9a
  - openai-text-embedding-3-smallab48d8d6-5167-570b-b380-f151c30b6577
  - openai-text-embedding-3-small2966e4a1-d89a-59bf-93f5-e19c970c43f6


---

## 마무리

### 핵심 요약

| 메서드 | 역할 | 사용 시점 |
|--------|------|-----------|
| `embed_query()` | 단일 텍스트 → 벡터 | 사용자 검색 질문 |
| `embed_documents()` | 여러 텍스트 → 벡터 배열 | 문서 색인(저장) |
| `dimensions` 파라미터 | 벡터 차원 축소 | 대용량 문서, 비용 절감 |
| `CacheBackedEmbeddings` | 임베딩 캐싱 | 운영 환경 필수 |

**운영 환경에서는 항상 `CacheBackedEmbeddings`를 사용하세요!** 반복 호출이 많은 RAG 시스템에서 비용을 크게 줄일 수 있어요.

### 다음 학습

**6-3 노트북 02**: HuggingFace 오픈소스 임베딩 모델 — API 비용 없이 로컬에서 임베딩을 생성하는 방법을 배워볼게요.